# Sale Price Prediction — Kaggle Notebook
**Created by Asya for Kaggle Competition**


Sections:
1. Data loading & overview
2. Target analysis
3. Missing values analysis
4. Feature engineering
5. Preprocessing and modeling
6. Final training and submission

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ===============================
# STEP 1: Robust data loading
# ===============================
import os
import pandas as pd

if os.path.exists("train.csv") and os.path.exists("test.csv"):
    train_df = pd.read_csv("train.csv")
    test_df = pd.read_csv("test.csv")
    print("Files loaded from local directory.")
else:
    try:
        train_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
        test_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")
        print("Files loaded from Kaggle Competition Dataset.")
    except FileNotFoundError:
        raise FileNotFoundError(
            "train.csv and test.csv not found. "
            "Download them from Kaggle or add them to the Kernel via 'Add Data'."
        )

print("🎯 BASIC DATA INFO:")
print("=" * 50)
print(f"Training set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")
print(f"Total features: {train_df.shape[1] - 1}")  # -1 because SalePrice is the target


print("\n📋 DETAILED DATA PREVIEW:")
print("=" * 50)

print("TRAINING DATA (first 3 rows):")
print("Shape:", train_df.shape)
print(train_df.head(3))
print("\n" + "-"*50)

print("TEST DATA (first 3 rows):")
print("Shape:", test_df.shape)
print(test_df.head(3))

In [ ]:
print("\n💰 TARGET (SalePrice) ANALYSIS:")
print("=" * 50)

print("SUMMARY STATISTICS:")
print(train_df['SalePrice'].describe())

print(f"Median price: ${train_df['SalePrice'].median():,.0f}")
print(f"Mean price: ${train_df['SalePrice'].mean():,.0f}")
print(f"Std dev: ${train_df['SalePrice'].std():,.0f}")
print(f"Range (max-min): ${train_df['SalePrice'].max() - train_df['SalePrice'].min():,.0f}")

# Plots: distribution, boxplot, log-distribution
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.histplot(train_df['SalePrice'], kde=True, bins=50)
plt.title('Distribution of SalePrice')
plt.xlabel('SalePrice ($)')
plt.ylabel('Number of houses')

plt.subplot(1, 3, 2)
sns.boxplot(y=train_df['SalePrice'])
plt.title('Boxplot of SalePrice')
plt.ylabel('SalePrice ($)')

plt.subplot(1, 3, 3)
sns.histplot(np.log1p(train_df['SalePrice']), kde=True, bins=50)
plt.title('Log-transformed SalePrice')
plt.xlabel('log(SalePrice + 1)')
plt.ylabel('Number of houses')

plt.tight_layout()
plt.show()

# Visual example showing difference between absolute and log
plt.figure(figsize=(15, 10))

sample_prices = [100000, 200000, 400000, 800000]
log_prices = np.log1p(sample_prices)

plt.subplot(2, 2, 1)
plt.bar(['$100K', '$200K', '$400K', '$800K'], sample_prices)
plt.title('ABSOLUTE PRICES\n(very large gap between $100K and $800K)')
plt.ylabel('Price ($)')

plt.subplot(2, 2, 2)
plt.bar(['$100K', '$200K', '$400K', '$800K'], log_prices)
plt.title('LOG-PRICES\n(differences look more even)')
plt.ylabel('log(Price + 1)')

plt.subplot(2, 2, 3)
sns.histplot(train_df['SalePrice'], bins=50, alpha=0.7)
plt.title('Original prices\n(long right tail)')
plt.xlabel('SalePrice ($)')

plt.subplot(2, 2, 4)
sns.histplot(np.log1p(train_df['SalePrice']), bins=50, alpha=0.7)
plt.title('Log-transformed prices\n(closer to normal)')
plt.xlabel('log(SalePrice + 1)')

plt.tight_layout()
plt.show()

In [ ]:
print("🔍 STEP 4: DETAILED MISSING VALUES ANALYSIS")
print("=" * 60)

train_clean = train_df.copy()
test_clean = test_df.copy()

def analyze_missing_data(df, dataset_name):
    print(f"\n📊 MISSING VALUES IN {dataset_name}:")
    print("-" * 40)
    total_cells = np.prod(df.shape)
    total_missing = df.isnull().sum().sum()
    missing_percent = (total_missing / total_cells) * 100
    print(f"Total cells: {total_cells:,}")
    print(f"Missing cells: {total_missing:,}")
    print(f"Overall missing %: {missing_percent:.2f}%")

    missing_data = df.isnull().sum()
    missing_percent_col = (missing_data / len(df)) * 100
    missing_df = pd.DataFrame({
        'Column': df.columns,
        'Missing': missing_data,
        'MissingPct': missing_percent_col
    })
    missing_df = missing_df[missing_df['Missing'] > 0].sort_values('MissingPct', ascending=False)
    print(f"Columns with missing: {len(missing_df)}")
    if len(missing_df) > 0:
        print("\n📋 TOP-15 COLUMNS WITH MOST MISSING VALUES:")
        print(missing_df.head(15).to_string(index=False))
    else:
        print("🎉 No missing values!")
    return missing_df

missing_train = analyze_missing_data(train_clean, 'TRAIN DATA')
missing_test = analyze_missing_data(test_clean, 'TEST DATA')

# Plot missing values summary if any
if len(missing_train) > 0:
    plt.figure(figsize=(15, 8))
    plt.subplot(2, 2, 1)
    top_missing_train = missing_train.head(20)
    sns.barplot(data=top_missing_train, y='Column', x='MissingPct')
    plt.title('TRAIN: Top-20 columns with missing values')
    plt.xlabel('Missing %')

    plt.subplot(2, 2, 2)
    top_missing_test = missing_test.head(20)
    sns.barplot(data=top_missing_test, y='Column', x='MissingPct')
    plt.title('TEST: Top-20 columns with missing values')
    plt.xlabel('Missing %')

    plt.subplot(2, 2, 3)
    comparison_data = pd.DataFrame({
        'Dataset': ['Train', 'Test'],
        'TotalCells': [np.prod(train_clean.shape), np.prod(test_clean.shape)],
        'Missing': [train_clean.isnull().sum().sum(), test_clean.isnull().sum().sum()]
    })
    sns.barplot(data=comparison_data, x='Dataset', y='Missing')
    plt.title('Total missing cells comparison')
    plt.ylabel('Missing cells')

    plt.tight_layout()
    plt.show()

print("\n🔎 ANALYSIS OF MISSING VALUE CAUSES:")
print("=" * 50)
print("Reading data_description.txt for domain knowledge if available...")

missing_categories = {
    'No object': ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
                  'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                  'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2'],
    'Zero value': ['MasVnrArea', 'GarageYrBlt', 'GarageArea', 'GarageCars',
                   'TotalBsmtSF', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
                   'BsmtFullBath', 'BsmtHalfBath'],
    'Actual missing': ['LotFrontage', 'MasVnrType', 'Electrical', 'Utilities',
                       'Functional', 'KitchenQual', 'SaleType', 'Exterior1st', 'Exterior2nd']
}

print('Missing categories (domain-based):')
for category, cols in missing_categories.items():
    existing = [c for c in cols if c in missing_train['Column'].values]
    if existing:
        print(f"\n{category}:")
        for col in existing:
            pct = missing_train[missing_train['Column'] == col]['MissingPct'].values[0]
            print(f"  - {col}: {pct:.1f}%")

# Recommend dropping very sparse columns
high_missing_threshold = 40
high_missing_cols = missing_train[missing_train['MissingPct'] > high_missing_threshold]
if len(high_missing_cols) > 0:
    print("\nColumns with more than 40% missing (candidates to drop):")
    for _, row in high_missing_cols.iterrows():
        print(f"  - {row['Column']}: {row['MissingPct']:.1f}%")

# Typical columns to drop for House Prices dataset
typical_columns_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence']
columns_to_drop_recommended = []
for col in typical_columns_to_drop:
    if col in missing_train['Column'].values:
        pct = missing_train[missing_train['Column'] == col]['MissingPct'].values[0]
        if pct > 80:
            columns_to_drop_recommended.append(col)
            print(f"Recommend drop: {col} ({pct:.1f}% missing)")

if columns_to_drop_recommended:
    train_clean.drop(columns_to_drop_recommended, axis=1, inplace=True)
    test_clean.drop(columns_to_drop_recommended, axis=1, inplace=True)
    print('Dropped columns:', columns_to_drop_recommended)
else:
    print('No high-missing candidates to drop')

In [ ]:
print("\n🔧 STEP 5: FILLING MISSING VALUES")
print("=" * 50)

fill_strategy = {
    'MasVnrType': 'None', 'Electrical': 'SBrkr', 'Utilities': 'AllPub',
    'FireplaceQu': 'None', 'GarageType': 'None', 'GarageFinish': 'None',
    'GarageQual': 'None', 'GarageCond': 'None', 'BsmtQual': 'None',
    'BsmtCond': 'None', 'BsmtExposure': 'None', 'BsmtFinType1': 'None',
    'BsmtFinType2': 'None',
    'MasVnrArea': 0, 'GarageYrBlt': 0, 'GarageArea': 0, 'GarageCars': 0,
    'TotalBsmtSF': 0, 'BsmtFinSF1': 0, 'BsmtFinSF2': 0, 'BsmtUnfSF': 0,
    'BsmtFullBath': 0, 'BsmtHalfBath': 0,
    'LotFrontage': 'median'
}

for col, val in fill_strategy.items():
    if col in train_clean.columns or col in test_clean.columns:
        train_miss_before = train_clean[col].isnull().sum() if col in train_clean.columns else 0
        test_miss_before = test_clean[col].isnull().sum() if col in test_clean.columns else 0
        if val == 'median' and col == 'LotFrontage':
            median_by_neighborhood = train_clean.groupby('Neighborhood')['LotFrontage'].median()
            if 'LotFrontage' in train_clean.columns:
                train_clean['LotFrontage'] = train_clean.apply(
                    lambda row: median_by_neighborhood[row['Neighborhood']] if pd.isna(row['LotFrontage']) else row['LotFrontage'],
                    axis=1
                )
            if 'LotFrontage' in test_clean.columns:
                test_clean['LotFrontage'] = test_clean.apply(
                    lambda row: median_by_neighborhood.get(row['Neighborhood'], train_clean['LotFrontage'].median()) if pd.isna(row['LotFrontage']) else row['LotFrontage'],
                    axis=1
                )
            print(f"  ✅ Filled {col} with neighborhood median")
        else:
            if col in train_clean.columns:
                train_clean[col].fillna(val, inplace=True)
            if col in test_clean.columns:
                test_clean[col].fillna(val, inplace=True)
            if train_miss_before > 0 or test_miss_before > 0:
                print(f"  ✅ {col}: filled {train_miss_before + test_miss_before} missing with '{val}'")

print('\nFilling complete!')

# Fill any remaining missing with mode/median
remaining_train_missing = train_clean.isnull().sum()
remaining_train_missing = remaining_train_missing[remaining_train_missing > 0]
remaining_test_missing = test_clean.isnull().sum()
remaining_test_missing = remaining_test_missing[remaining_test_missing > 0]

all_remaining = set(remaining_train_missing.index) | set(remaining_test_missing.index)
for col in all_remaining:
    if col in train_clean.columns:
        if train_clean[col].dtype == 'object':
            mode_val = train_clean[col].mode()[0] if len(train_clean[col].mode()) > 0 else 'None'
            train_clean[col].fillna(mode_val, inplace=True)
            if col in test_clean.columns:
                test_clean[col].fillna(mode_val, inplace=True)
            print(f"  ✅ {col}: filled categorical with mode '{mode_val}'")
        else:
            median_val = train_clean[col].median()
            train_clean[col].fillna(median_val, inplace=True)
            if col in test_clean.columns:
                test_clean[col].fillna(median_val, inplace=True)
            print(f"  ✅ {col}: filled numeric with median {median_val}")

print('\nFinal missing check:')
print('Train missing:', train_clean.isnull().sum().sum())
print('Test missing:', test_clean.isnull().sum().sum())

In [ ]:
print('\n🔧 STEP 6: FEATURE ENGINEERING')
print('=' * 35)

train_fe = train_clean.copy()
test_fe = test_clean.copy()

train_fe['TotalSF'] = train_fe['TotalBsmtSF'] + train_fe['1stFlrSF'] + train_fe['2ndFlrSF']
test_fe['TotalSF'] = test_fe['TotalBsmtSF'] + test_fe['1stFlrSF'] + test_fe['2ndFlrSF']
print('TotalSF created')

train_fe['TotalBath'] = (train_fe['FullBath'] + train_fe['BsmtFullBath'] + 0.5 * train_fe['HalfBath'] + 0.5 * train_fe['BsmtHalfBath'])
test_fe['TotalBath'] = (test_fe['FullBath'] + test_fe['BsmtFullBath'] + 0.5 * test_fe['HalfBath'] + 0.5 * test_fe['BsmtHalfBath'])
print('TotalBath created')

train_fe['HouseAge'] = train_fe['YrSold'] - train_fe['YearBuilt']
test_fe['HouseAge'] = test_fe['YrSold'] - test_fe['YearBuilt']
train_fe['YearsSinceRemod'] = train_fe['YrSold'] - train_fe['YearRemodAdd']
test_fe['YearsSinceRemod'] = test_fe['YrSold'] - test_fe['YearRemodAdd']
print('HouseAge and YearsSinceRemod created')

train_fe['OverallQual_TotalSF'] = train_fe['OverallQual'] * train_fe['TotalSF']
test_fe['OverallQual_TotalSF'] = test_fe['OverallQual'] * test_fe['TotalSF']
train_fe['SFPerRoom'] = train_fe['TotalSF'] / train_fe['TotRmsAbvGrd']
test_fe['SFPerRoom'] = test_fe['TotalSF'] / test_fe['TotRmsAbvGrd']
train_fe['BedroomRatio'] = train_fe['BedroomAbvGr'] / train_fe['TotRmsAbvGrd']
test_fe['BedroomRatio'] = test_fe['BedroomAbvGr'] / test_fe['TotRmsAbvGrd']
train_fe['LotRatio'] = train_fe['LotArea'] / train_fe['TotalSF']
test_fe['LotRatio'] = test_fe['LotArea'] / test_fe['TotalSF']
print('Additional engineered features created')

new_feature_list = ['TotalSF', 'TotalBath', 'HouseAge', 'YearsSinceRemod', 'OverallQual_TotalSF', 'SFPerRoom', 'BedroomRatio', 'LotRatio']

# Correlation checks
corr_summary = []
for f in new_feature_list:
    if f in train_fe.columns:
        corr_summary.append((f, train_fe[f].corr(train_fe['SalePrice'])))

corr_df_simple = pd.DataFrame(corr_summary, columns=['Feature', 'Correlation']).sort_values('Correlation', ascending=False)
print('\nCorrelation of new features with SalePrice:')
print(corr_df_simple)

In [ ]:
print('\n🔧 STEP 7: PREPARE DATA FOR MODELING')

y = train_fe['SalePrice']
X = train_fe.drop(['SalePrice'], axis=1)
X_test = test_fe.copy()

print(f'X shape: {X.shape}, y shape: {y.shape}, X_test shape: {X_test.shape}')

y_log = np.log1p(y)
sns.histplot(y_log, kde=True, color='orange')
plt.title('Log-transformed target distribution')
plt.show()

categorical_cols = X.select_dtypes(include=['object']).columns
numerical_cols = X.select_dtypes(exclude=['object']).columns
print(f'Categorical cols: {len(categorical_cols)}')
print(f'Numeric cols: {len(numerical_cols)}')

# Preprocessor pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(X, y_log, test_size=0.2, random_state=42)

X_train_prepared = preprocessor.fit_transform(X_train)
X_valid_prepared = preprocessor.transform(X_valid)
X_test_prepared = preprocessor.transform(X_test)

print('Preprocessing complete')
print('Encoded train shape:', X_train_prepared.shape)

In [ ]:
print('\n🔧 STEP 8: MODEL TRAINING & EVALUATION')

from sklearn.metrics import mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print('\nLinear Regression:')
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train_prepared, y_train)
y_pred_lr = lr.predict(X_valid_prepared)
print('LR RMSE:', rmse(y_valid, y_pred_lr))

print('\nRandom Forest Regressor:')
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train_prepared, y_train)
y_pred_rf = rf.predict(X_valid_prepared)
print('RF RMSE:', rmse(y_valid, y_pred_rf))

print('\nCatBoost Regressor:')
from catboost import CatBoostRegressor
cat = CatBoostRegressor(iterations=2000, learning_rate=0.05, depth=6, loss_function='RMSE', verbose=False, random_seed=42)
cat.fit(X_train_prepared, y_train)
y_pred_cat = cat.predict(X_valid_prepared)
print('CatBoost RMSE:', rmse(y_valid, y_pred_cat))

In [ ]:
print('\n🔧 STEP 9: FINAL TRAINING AND SUBMISSION')

# Train preprocessor on full train+test to avoid unseen categories
X_full = pd.concat([X, X_test], axis=0)
preprocessor.fit(X_full)
X_all_prepared = preprocessor.transform(X)

# Final CatBoost model
final_model = CatBoostRegressor(iterations=2000, learning_rate=0.05, depth=6, loss_function='RMSE', verbose=False, random_seed=42)
final_model.fit(X_all_prepared, np.log1p(y))

X_test_prepared = preprocessor.transform(X_test)
test_preds = final_model.predict(X_test_prepared)
test_preds_final = np.expm1(test_preds)

submission = pd.DataFrame({'Id': test_fe['Id'], 'SalePrice': test_preds_final})
submission.to_csv('submission.csv', index=False)
print('Submission file saved: submission.csv')

In [ ]:
print('\n🔧 STEP 10: TRAIN/VAL METRICS (MAPE & Accuracy)')
from sklearn.model_selection import train_test_split
# Split again for validation metrics (already done earlier, reuse variables)

# Define MAPE that handles log target
def mean_absolute_percentage_error_log(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Validation metrics
val_preds = final_model.predict(preprocessor.transform(X_valid))
val_mape = mean_absolute_percentage_error_log(y_valid, val_preds)
val_accuracy = 100 - val_mape

# Train metrics
train_preds = final_model.predict(X_all_prepared)
train_mape = mean_absolute_percentage_error_log(np.log1p(y), train_preds)
train_accuracy = 100 - train_mape

metrics_df = pd.DataFrame({'Dataset': ['Train', 'Validation'], 'MAPE (%)': [train_mape, val_mape], 'Accuracy (%)': [train_accuracy, val_accuracy]})
print('\nModel performance:')
print(metrics_df)

# Notebook end
print('\nGood luck on Kaggle!')